# HypotheSAEs on Hippocorpus: what distinguishes recalled from imagined stories?

Runs the HypotheSAEs pipeline (embed → SAE → select neurons → interpret → evaluate) on **Hippocorpus** (Sap et al., ACL 2020): ~5.5K first-person stories that people either **recalled** from their own lives or **imagined** from a short summary.

Setup used here:
- **Embeddings + SAE**: run in this Colab session (use a GPU runtime: *Runtime → Change runtime type → T4 GPU*).
- **LLM (interpretation + annotation)**: Qwen3-8B served with vLLM on Modal as an OpenAI-compatible endpoint (`modal/vllm_server.py`).
- Annotation uses `prompt_suffix=" /no_think"` (added in this fork) to turn off Qwen3 thinking for yes/no calls; interpretation keeps thinking on.

Before running: add a Colab secret (key icon in the left sidebar) named `MODAL_VLLM_KEY` with the server's API key, and enable notebook access.

In [ ]:
!pip install -q git+https://github.com/psnandini/HypotheSAEs.git
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_BASE_URL"] = "https://psnandini--hypothesaes-vllm-serve.us-east.modal.run/v1"
os.environ["OPENAI_KEY_SAE"] = userdata.get("MODAL_VLLM_KEY")
os.environ["EMB_CACHE_DIR"] = "/content/emb_cache"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hypothesaes.quickstart import train_sae, interpret_sae, generate_hypotheses, evaluate_hypotheses
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.llm_api import get_completion

MODEL = "Qwen/Qwen3-8B"
INTERPRETER_MODEL = ANNOTATOR_MODEL = MODEL

**Wake up the LLM server.** The Modal server scales to zero when idle, so the first call can take 2–3 minutes while a GPU starts. Later calls are fast.

In [ ]:
import time
t = time.time()
print(get_completion(prompt="Reply with the single word: ready /no_think", model=MODEL))
print(f"{time.time() - t:.0f} s")

## 0. Load Hippocorpus

Downloaded from the official Microsoft source (Open Use of Data Agreement). We keep **recalled** and **imagined** stories; label = 1 for recalled, 0 for imagined.

Each imagined story was written from the summary of a recalled story (`recImgPairId`), so the two share a topic. We keep each pair in the same split so that topic cannot leak between train and held-out data.

In [ ]:
!wget -q -nc https://download.microsoft.com/download/3/c/3/3c388755-ac68-4858-8343-9acfb33c631d/hippocorpus-u20220112.zip
!unzip -q -o hippocorpus-u20220112.zip -d hippocorpus

df = pd.read_csv("hippocorpus/hcV3-stories.csv", encoding="latin-1")
df = df[df.memType.isin(["recalled", "imagined"])].dropna(subset=["story"]).reset_index(drop=True)
df["label"] = (df.memType == "recalled").astype(int)
# pair id: a recalled story's own id, or the recalled story an imagined one was based on
df["pair_id"] = np.where(df.memType == "recalled", df.AssignmentId, df.recImgPairId.fillna(df.AssignmentId))
df["n_words"] = df.story.str.split().str.len()
print(df.memType.value_counts())

In [ ]:
rng = np.random.default_rng(0)
pairs = rng.permutation(np.asarray(df.pair_id.unique(), dtype=object))
n = len(pairs)
split_of = {p: "train" for p in pairs[: int(0.7 * n)]}
split_of.update({p: "val" for p in pairs[int(0.7 * n): int(0.8 * n)]})
split_of.update({p: "holdout" for p in pairs[int(0.8 * n):]})
df["split"] = df.pair_id.map(split_of)
print(df.groupby(["split", "memType"]).size().unstack())

train_df, val_df, holdout_df = (df[df.split == s].reset_index(drop=True) for s in ["train", "val", "holdout"])
texts, labels = train_df.story.tolist(), train_df.label.values
val_texts = val_df.story.tolist()

Quick check: story length by class. If length alone separated the classes strongly, many "hypotheses" could just be length proxies.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
for name, g in df.groupby("memType"):
    ax.hist(g.n_words, bins=50, alpha=0.5, label=f"{name} (median {g.n_words.median():.0f})")
ax.set_xlabel("words per story"); ax.set_ylabel("stories"); ax.legend(); plt.tight_layout(); plt.show()

## 1. Text embeddings

`nomic-ai/modernbert-embed-base`, the local embedder used in the HypotheSAEs local quickstart. Embeddings are cached in `EMB_CACHE_DIR`.

In [ ]:
EMBEDDER = "nomic-ai/modernbert-embed-base"
CACHE_NAME = "hippocorpus_modernbert"

text2embedding = get_local_embeddings(texts + val_texts, model=EMBEDDER, batch_size=64, cache_name=CACHE_NAME)
train_embeddings = np.stack([text2embedding[t] for t in texts])
val_embeddings = np.stack([text2embedding[t] for t in val_texts])
print(train_embeddings.shape, val_embeddings.shape)

## 2. Train the SAE

Same settings as the quickstart: Matryoshka SAE with M=256 neurons, K=8 active per text, prefixes [32, 256]. Hippocorpus is small (~3.9K training stories), so check the reconstruction and dead-neuron numbers printed during training; a smaller M is worth trying if many neurons are dead.

In [ ]:
sae = train_sae(
    embeddings=train_embeddings, M=256, K=8, matryoshka_prefix_lengths=[32, 256],
    checkpoint_dir="/content/checkpoints/hippocorpus", val_embeddings=val_embeddings,
)

## 3. Sanity check: interpret a few random neurons

In [ ]:
TASK_SPECIFIC_INSTRUCTIONS = """All of the texts are short first-person stories written by crowdworkers about an event in their life.
Features should describe a specific property of the story's content or writing style. For example:
- "mentions a pet or animal"
- "describes a conversation with a family member"
"""

INTERPRET_LLM_KWARGS = {}                              # thinking on for interpretation
ANNOTATION_LLM_KWARGS = {"prompt_suffix": " /no_think"}  # thinking off for yes/no annotation

random_results = interpret_sae(
    texts=texts, embeddings=train_embeddings, sae=sae,
    interpreter_model=INTERPRETER_MODEL, n_random_neurons=5, print_examples_n=3,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS, interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)

## 4. Generate hypotheses

Select the 20 neurons most correlated with the label, interpret each, and score each interpretation's fidelity (F1 between the neuron firing and the LLM's annotation of the concept).

In [ ]:
results = generate_hypotheses(
    texts=texts, labels=labels, embeddings=train_embeddings, sae=sae,
    interpreter_model=INTERPRETER_MODEL, annotator_model=ANNOTATOR_MODEL,
    selection_method="correlation", n_selected_neurons=20, n_candidate_interpretations=1,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS, annotation_llm_kwargs=ANNOTATION_LLM_KWARGS,
    n_workers_interpretation=8, n_workers_annotation=32,
)
pd.set_option("display.max_colwidth", None)
display(results.sort_values("target_correlation", ascending=False))
results.to_csv("hippocorpus_hypotheses.csv", index=False)

## 5. Evaluate on held-out stories

Annotate each hypothesis on held-out stories (pairs never seen during SAE training or neuron selection) and test whether it predicts recalled vs imagined.

In [ ]:
metrics, evaluation_df = evaluate_hypotheses(
    hypotheses_df=results, texts=holdout_df.story.tolist(), labels=holdout_df.label.values,
    annotator_model=ANNOTATOR_MODEL, annotation_llm_kwargs=ANNOTATION_LLM_KWARGS, n_workers_annotation=32,
)
display(evaluation_df)
evaluation_df.to_csv("hippocorpus_holdout_eval.csv", index=False)
print(f"AUROC: {metrics['auroc']:.3f}")
print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} (p < {metrics['Significant'][2]})")

In [ ]:
from google.colab import files
files.download("hippocorpus_hypotheses.csv")
files.download("hippocorpus_holdout_eval.csv")